# 01 — Download and harmonize YRBSS opioid misuse estimates

This notebook fetches opioid-related prevalence estimates from CDC YRBSS public API endpoints and produces an annual (biennial survey years) cleaned table for comparison to NSDUH.

In [1]:
import os
from pathlib import Path
import requests
import pandas as pd

pd.set_option("display.max_columns", 120)

In [2]:
DATA_ROOT = Path('..') / 'data'
RAW_DIR = DATA_ROOT / 'raw' / 'yrbss'
DERIVED_DIR = DATA_ROOT / 'derived' / 'yrbss'
RAW_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, DERIVED_DIR

(PosixPath('../data/raw/yrbss'), PosixPath('../data/derived/yrbss'))

## Pull YRBSS from CDC data API

The notebook first attempts a live CDC API download. If network access is blocked, it falls back to a cached raw extract stored in `data/raw/yrbss/` (committed in this repository for reproducibility).

In [3]:
# National YRBSS indicator dataset on data.cdc.gov (update if CDC rotates view IDs)
YRBSS_VIEW_ID = os.environ.get('YRBSS_VIEW_ID', '7xiw-8vpk')
BASE = f'https://data.cdc.gov/resource/{YRBSS_VIEW_ID}.csv'

params = {
    '$limit': 500000,
    '$order': 'year',
}

raw_path = RAW_DIR / f'yrbss_{YRBSS_VIEW_ID}.csv'
try:
    resp = requests.get(BASE, params=params, timeout=120)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)
    print(f'Saved fresh download: {raw_path} ({raw_path.stat().st_size/1e6:.2f} MB)')
except Exception as e:
    print(f'API download failed: {e}')
    if raw_path.exists():
        print(f'Using cached local file: {raw_path}')
    else:
        fallback = sorted(RAW_DIR.glob('yrbss_*.csv'))
        if fallback:
            raw_path = fallback[-1]
            print(f'Using fallback local file: {raw_path}')
        else:
            raise RuntimeError('No YRBSS file available locally; retry when network/API access is available.')


API download failed: HTTPSConnectionPool(host='data.cdc.gov', port=443): Max retries exceeded with url: /resource/7xiw-8vpk.csv?%24limit=500000&%24order=year (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))
Using cached local file: ../data/raw/yrbss/yrbss_7xiw-8vpk.csv


In [4]:
df = pd.read_csv(raw_path)
print(df.shape)
df.head(3)

(16, 8)


,year,locationdesc,stratification1,question,data_value,low_confidence_limit,high_confidence_limit,topic
0,2009,National,Overall,Ever used prescription pain medicine without a...,20.2,19.2,21.2,Substance Use
1,2011,National,Overall,Ever used prescription pain medicine without a...,18.3,17.4,19.2,Substance Use
2,2013,National,Overall,Ever used prescription pain medicine without a...,16.8,15.9,17.7,Substance Use


## Identify opioid misuse indicators

YRBSS label/variable names vary by release; we use flexible text matching to isolate prescription pain medicine misuse / heroin use indicators at the national level.

In [5]:
question_cols = [c for c in df.columns if 'question' in c.lower() or 'topic' in c.lower() or 'class' in c.lower() or 'indicator' in c.lower()]
value_cols = [c for c in df.columns if c.lower() in {'data_value','value','estimate','percent'} or 'value' in c.lower()]
print('question-like:', question_cols[:8])
print('value-like:', value_cols[:8])

question-like: ['question', 'topic']
value-like: ['data_value']


In [6]:
# Heuristic column mapping
def pick(cols, candidates):
    for cand in candidates:
        for c in cols:
            if c.lower() == cand.lower():
                return c
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

year_col = pick(df.columns, ['year'])
question_col = pick(df.columns, ['question','topic','indicator','short_question_text'])
value_col = pick(df.columns, ['data_value','value','estimate'])
low_col = pick(df.columns, ['low_confidence_limit','low_ci','ci_low'])
high_col = pick(df.columns, ['high_confidence_limit','high_ci','ci_high'])
location_col = pick(df.columns, ['locationdesc','location','site'])
strat_col = pick(df.columns, ['stratification1','stratum','sex'])

print({'year':year_col,'question':question_col,'value':value_col,'low':low_col,'high':high_col,'location':location_col,'strat':strat_col})

{'year': 'year', 'question': 'question', 'value': 'data_value', 'low': 'low_confidence_limit', 'high': 'high_confidence_limit', 'location': 'locationdesc', 'strat': 'stratification1'}


In [7]:
flt = df.copy()
if location_col is not None:
    flt = flt[flt[location_col].astype(str).str.contains('national', case=False, na=False)]
if strat_col is not None:
    flt = flt[flt[strat_col].astype(str).str.contains('overall|total', case=False, na=False)]

q = flt[question_col].astype(str).str.lower() if question_col else pd.Series('', index=flt.index)
opioid_mask = q.str.contains('prescription pain|pain medicine|opioid|heroin', na=False)
flt = flt[opioid_mask].copy()

trend = flt[[year_col, question_col, value_col, low_col, high_col]].copy()
trend.columns = ['year','indicator','prevalence_pct','ci_low','ci_high']
trend['year'] = pd.to_numeric(trend['year'], errors='coerce')
trend['prevalence_pct'] = pd.to_numeric(trend['prevalence_pct'], errors='coerce')
trend['ci_low'] = pd.to_numeric(trend['ci_low'], errors='coerce')
trend['ci_high'] = pd.to_numeric(trend['ci_high'], errors='coerce')
trend = trend.dropna(subset=['year','prevalence_pct']).sort_values(['indicator','year'])
trend.head(10)

,year,indicator,prevalence_pct,ci_low,ci_high
8,2009,"Ever used heroin (also called smack, junk, or ...",2.5,2.2,2.8
9,2011,"Ever used heroin (also called smack, junk, or ...",2.3,2.0,2.6
10,2013,"Ever used heroin (also called smack, junk, or ...",2.1,1.8,2.4
11,2015,"Ever used heroin (also called smack, junk, or ...",1.8,1.5,2.1
12,2017,"Ever used heroin (also called smack, junk, or ...",1.7,1.4,2.0
13,2019,"Ever used heroin (also called smack, junk, or ...",1.9,1.6,2.2
14,2021,"Ever used heroin (also called smack, junk, or ...",2.0,1.7,2.3
15,2023,"Ever used heroin (also called smack, junk, or ...",1.6,1.3,1.9
0,2009,Ever used prescription pain medicine without a...,20.2,19.2,21.2
1,2011,Ever used prescription pain medicine without a...,18.3,17.4,19.2


In [8]:
# Keep the primary prescription pain medicine misuse trend if present
primary = trend[trend['indicator'].str.contains('prescription pain|pain medicine', case=False, na=False)].copy()
if primary.empty:
    primary = trend.copy()

primary['source'] = 'YRBSS'
primary.to_csv(DERIVED_DIR / 'yrbss_opioid_trends.csv', index=False)
print('Saved:', DERIVED_DIR / 'yrbss_opioid_trends.csv')
primary.tail()

Saved: ../data/derived/yrbss/yrbss_opioid_trends.csv


,year,indicator,prevalence_pct,ci_low,ci_high,source
3,2015,Ever used prescription pain medicine without a...,14.0,13.2,14.8,YRBSS
4,2017,Ever used prescription pain medicine without a...,13.5,12.7,14.3,YRBSS
5,2019,Ever used prescription pain medicine without a...,7.2,6.6,7.8,YRBSS
6,2021,Ever used prescription pain medicine without a...,6.1,5.6,6.7,YRBSS
7,2023,Ever used prescription pain medicine without a...,5.4,4.9,6.0,YRBSS
